# Rategoan - Lanjutkan Training Neural 50M / 100M di GPU T4 gratis (Colab)

**MEGA-BATCH RAGETAN ROUND 5 FASE 2 + ROUND 6 FASE 2/3.** Notebook ini melanjutkan training checkpoint `raget-neural-massive50m.safetensors` (50M parameter) ATAU melatih model BARU `raget-neural-massive100m.safetensors` (~103,3 juta parameter, arsitektur lebih besar, vocab BPE 30.368 yang SAMA dipakai ulang - "satu jiwa, dua badan") - di GPU T4 gratis Colab, bukan CPU sandbox. Pilih ukuran model di sel 8 (`MODEL_SIZE`).

## 3 LANGKAH MANUAL (yang lain otomatis)

1. **Runtime > Change runtime type > T4 GPU** (lalu Save).
2. Klik ikon kunci (Secrets) di sidebar kiri Colab, tambahkan secret bernama **`GITHUB_TOKEN`** berisi GitHub Personal Access Token (scope `repo`, dipakai untuk clone repo private + unduh aset Release + push hasil balik). Kalau tidak diset, sel di bawah akan minta paste token manual sekali.
3. **Runtime > Run all.** (opsional: ubah `MODEL_SIZE` di sel 8 ke `'100m'` sebelum Run all kalau mau melatih model besar, default `'50m'`)

Semua sel di bawah (unduh dump Wikipedia dari GitHub Release 'Korpus', ekstrak+bersihkan, tokenisasi ulang memakai tokenizer YANG SAMA PERSIS dari metadata checkpoint 50M (bukan dilatih ulang - supaya token ID tidak berubah, dipakai baik untuk 50M maupun 100M), training GPU fp16, lalu commit+push checkpoint & laporan balik ke GitHub) berjalan otomatis tanpa input lanjutan.

Preprocessing (unduh dump 1,15GB + extract + tokenisasi) makan waktu sekitar 15-25 menit CPU Colab sebelum GPU mulai dipakai untuk training - satu kali di awal, bukan berulang (dipakai bersama baik untuk 50M maupun 100M).

In [ ]:
# @title 0. Cek GPU tersedia
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print('torch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: tidak ada GPU terdeteksi - pastikan Runtime > Change runtime type > T4 GPU sudah dipilih (langkah manual 1).')

In [ ]:
# @title 1. Ambil GitHub token (Secret Colab atau input manual)
import os
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
if not GITHUB_TOKEN:
    import getpass
    GITHUB_TOKEN = getpass.getpass('Secret GITHUB_TOKEN tidak ditemukan - paste GitHub PAT (scope repo) di sini: ')
assert GITHUB_TOKEN, 'Token GitHub wajib diisi untuk clone repo private + unduh aset Release + push hasil.'
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
print('Token diterima (', len(GITHUB_TOKEN), 'karakter).')

In [ ]:
# @title 2. Clone repo Rategoan
REPO_OWNER = 'maetalizer-png'
REPO_NAME = 'Rategoan'
BRANCH = 'main'  # @param {type:"string"}

clone_url = 'https://' + GITHUB_TOKEN + '@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git'
!rm -rf /content/repo
!git clone --branch {BRANCH} --single-branch {clone_url} /content/repo
%cd /content/repo
!git config user.email 'colab-gpu-runner@rategoan.local'
!git config user.name 'Rategoan Colab GPU Runner'
!ls raget/raget-data/neural/
!ls raget/raget-tools/train-massive50m-colab-gpu.py

In [ ]:
# @title 3. Unduh dump Wikipedia dari GitHub Release 'Korpus' (autentikasi PAT - repo private)
import requests, os

ASSET_FILENAME = 'idwiki-latest-pages-articles.xml.bz2'
TAG = 'Korpus'

meta_headers = {'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/vnd.github+json'}
r = requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/tags/' + TAG, headers=meta_headers)
r.raise_for_status()
release = r.json()
asset = next(a for a in release['assets'] if a['name'] == ASSET_FILENAME)
asset_id = asset['id']
print('Aset ditemukan: id=', asset_id, 'ukuran=', asset['size'], 'bytes')

dl_headers = {'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/octet-stream'}
dump_path = '/content/idwiki-latest-pages-articles.xml.bz2'
with requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/assets/' + str(asset_id), headers=dl_headers, stream=True, allow_redirects=True) as resp:
    resp.raise_for_status()
    with open(dump_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)
print('Selesai unduh:', os.path.getsize(dump_path), 'bytes (harus mendekati', asset['size'], ')')

In [ ]:
# @title 4. Ekstrak tokenizer (merges+vocab) LANGSUNG dari metadata checkpoint
# Token ID WAJIB identik dengan yang dipakai waktu checkpoint ini dilatih -
# bpe-tokenizer.json terpisah TIDAK dikomit ke git (lihat wikipedia-korpus-manifest.json),
# tapi merges+vocabEntries-nya tersimpan ulang di metadata checkpoint sendiri.
!python3 raget/raget-tools/extract-tokenizer-from-checkpoint.py raget/raget-data/neural/raget-neural-massive50m.safetensors /content/bpe-tokenizer.json
!ls -la /content/bpe-tokenizer.json

In [ ]:
# @title 5. Extract + bersihkan wikitext (regenerasi korpus - deterministik, sama seperti sandbox)
STOP_AT_WORDS = 150000000  # @param {type:"integer"}
!python3 raget/raget-tools/extract-clean-wikipedia.py /content/idwiki-latest-pages-articles.xml.bz2 /content/idwiki-clean.jsonl {STOP_AT_WORDS}

In [ ]:
# @title 6. Tokenisasi + chunking (window 512) memakai tokenizer checkpoint
!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/idwiki-clean.jsonl /content/bpe-tokenizer.json /content/idwiki-chunks.txt 512

In [ ]:
# @title 7. Rechunk ke window 126 (sama seperti round 3/4/5 CPU - attention lebih murah)
!python3 raget/raget-tools/rechunk-corpus-window.py /content/idwiki-chunks.txt /content/idwiki-chunks-126.txt 126

In [ ]:
# @title 8. TRAINING GPU (fp16 autocast + GradScaler, T4) - pilih 50M atau 100M
MODEL_SIZE = '50m'  # @param ['50m', '100m']
BUDGET_MINUTES = 300  # @param {type:"number"}
BATCH_SIZE = 128  # @param {type:"number"}

if MODEL_SIZE == '100m':
    # Model BARU (fresh init pertama kali, resume otomatis kalau checkpoint
    # 100M sudah ada dari run sebelumnya) - vocab dipinjam dari checkpoint
    # 50M yang ikut ter-clone bersama repo.
    !python3 raget/raget-tools/train-massive100m-colab-gpu.py {BUDGET_MINUTES} {BATCH_SIZE} /content/idwiki-chunks-126.txt raget/raget-data/neural/raget-neural-massive50m.safetensors
else:
    !python3 raget/raget-tools/train-massive50m-colab-gpu.py {BUDGET_MINUTES} {BATCH_SIZE} /content/idwiki-chunks-126.txt

In [ ]:
# @title 9. Commit + push checkpoint & laporan balik ke GitHub (branch terpisah, aman dari bentrok)
PUSH_BRANCH = 'colab-gpu-training'  # @param {type:"string"}

if MODEL_SIZE == '100m':
    CKPT_FILE = 'raget/raget-data/neural/raget-neural-massive100m.safetensors'
    REPORT_FILE = 'raget/raget-devlog/neural/training-report-massive100m-round6-colab-gpu.json'
else:
    CKPT_FILE = 'raget/raget-data/neural/raget-neural-massive50m.safetensors'
    REPORT_FILE = 'raget/raget-devlog/neural/training-report-massive50m-round5-colab-gpu.json'

!git add {CKPT_FILE} {REPORT_FILE}
!git commit -m "Round 5/6: training GPU Colab T4 ({MODEL_SIZE}) lanjutan/baru checkpoint massive{MODEL_SIZE}" || echo "(tidak ada perubahan untuk di-commit)"
!git push origin HEAD:{PUSH_BRANCH}
print('Selesai. Checkpoint (' + MODEL_SIZE + ') + laporan sudah di-push ke branch:', PUSH_BRANCH)
print('Merge branch ini ke main lewat Pull Request di GitHub, atau minta sesi Rategoan berikutnya untuk merge otomatis.')

## Catatan jujur

- Sel 5-7 (extract+clean, tokenisasi, rechunk) meregenerasi korpus dari dump Wikipedia yang sama secara DETERMINISTIK (skrip identik dengan yang dipakai di sandbox CPU round 3), bukan mengunduh korpus siap-pakai - ini satu-satunya cara membawa korpus ~1GB ke Colab tanpa upload manual, karena file hasil tokenisasi terlalu besar untuk dikomit ke git.
- Token/detik GPU T4 yang sebenarnya HANYA diketahui setelah sel 8 selesai jalan - lihat output sel tersebut dan `training-report-massive50m-round5-colab-gpu.json` untuk angka nyata (field `tokenPerSecAvg`), bukan estimasi dari notebook ini.
- fp16 (bukan bf16) dipakai karena GPU T4 = arsitektur Turing, tidak punya tensor core bf16 yang baik seperti Ampere+ - fp16 + `GradScaler` adalah pendekatan standar untuk T4.
- Checkpoint hasil Colab di-push ke branch terpisah (`colab-gpu-training` secara default) supaya tidak bentrok dengan commit yang sedang berjalan di sandbox lain - perlu di-merge manual (PR) atau oleh sesi Rategoan berikutnya.